In [1]:
from ultralytics import YOLO
import torch
import cv2
import glob
import os

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


In [4]:
# download the dataset
from roboflow import Roboflow
rf = Roboflow(api_key="QpZpREjqsznAKm43We43")
project = rf.workspace("frisbee-cv").project("frisbee-cv-90zsv")
version = project.version(1)
dataset = version.download("yolo26")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Frisbee-CV-1 in yolo26:: 100%|██████████| 11718/11718 [00:11<00:00, 995.78it/s] 


In [6]:
model = YOLO("yolo26s.pt")

# train the model
model.train(
    data="../data/data.yaml",
    epochs=100,
    patience = 50,
    imgsz=640,
    batch=16,
    device='0')  # specify the GPU device, e.g., "0" for the first GPU


Ultralytics 8.4.27  Python-3.13.3 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce GTX 1060 6GB, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=50, perspective

KeyboardInterrupt: 

In [ ]:
# save the model
model.save("yolo26s_trained.pt")

# test the model
predictions = model.predict(
    source="../data/test/images",
    conf=0.25,
    save=True,
    save_txt=True,
    save_conf=True,
    device=device
)



In [ ]:
# 1) Load your trained model
model = YOLO("yolo26s_trained.pt")

# 2) Run inference (no need for save=True)
results = model.predict(
    source="../data/test/images",
    conf=0.25,
    device="cuda"   # or "cpu"
)

# 3) Prepare VideoWriter using the size of the first plotted frame
first_annotated = results[0].plot()               # returns BGR numpy array
h, w, _ = first_annotated.shape
fps = 2                                          # adjust to taste
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter("yolo26s_test_output.mp4", fourcc, fps, (w, h))

# 4) Loop through all results, plot each, and write to the video
for r in results:
    annotated_frame = r.plot()   # draws boxes/labels on a copy of the frame
    out.write(annotated_frame)

out.release()
print("✅ Saved annotated video as yolo11m_test_output.mp4")



image 1/202 c:\Users\BurnD\Desktop\Flying_plate_tracker\yolo26_joel\training\..\data\test\images\-BH1-_frame108_jpg.rf.72fed3f108e02228d1a5311d0222a81c.jpg: 640x640 1 frisbee, 53.3ms
image 2/202 c:\Users\BurnD\Desktop\Flying_plate_tracker\yolo26_joel\training\..\data\test\images\-BH1-_frame122_jpg.rf.ab8af3756da8c852135e6660b72d6002.jpg: 640x640 1 frisbee, 56.2ms
image 3/202 c:\Users\BurnD\Desktop\Flying_plate_tracker\yolo26_joel\training\..\data\test\images\-BH1-_frame128_jpg.rf.e3ea80b16de44ca3d7c2c879977a80dd.jpg: 640x640 1 frisbee, 41.5ms
image 4/202 c:\Users\BurnD\Desktop\Flying_plate_tracker\yolo26_joel\training\..\data\test\images\-BH1-_frame129_jpg.rf.8c6ec5faa2c325258aa966d6eb62ff44.jpg: 640x640 1 frisbee, 27.9ms
image 5/202 c:\Users\BurnD\Desktop\Flying_plate_tracker\yolo26_joel\training\..\data\test\images\-BH1-_frame140_jpg.rf.adf4638e1d34b35dd28963ca4ac2eadd.jpg: 640x640 1 frisbee, 27.7ms
image 6/202 c:\Users\BurnD\Desktop\Flying_plate_tracker\yolo26_joel\training\..\data

In [14]:
import os
import imageio
import cv2
from ultralytics import YOLO

# 1) Load your trained model
model = YOLO("yolo26s_trained.pt")

# 2) Read the source GIF into a list of frames (as RGB arrays)
gif_path    = "../edge_test/waitforit-wait.gif"
frames_rgb  = imageio.mimread(gif_path, memtest=False)

# 3) Prepare a list to collect annotated frames
annotated = []

# 4) Loop over each frame
for i, frame in enumerate(frames_rgb):
    # YOLO expects BGR on CPU/GPU, so convert:
    frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

    # Run a single-image prediction (batch of 1)
    results = model.predict(
        source=frame_bgr,
        conf=0.1,
        save=False,
        device="cuda"  # or "cpu"
    )

    # results is a list of length 1; get the plotted BGR image
    out_bgr = results[0].plot()

    # Convert back to RGB for GIF
    out_rgb = cv2.cvtColor(out_bgr, cv2.COLOR_BGR2RGB)
    annotated.append(out_rgb)

# 5) Write annotated frames back out as a new GIF
output_gif = "../edge_test/waitforit-wait-annotated.gif"
imageio.mimsave(output_gif, annotated, fps=3)   # tweak fps as desired
print(f"✅ Annotated GIF written to {output_gif}")



0: 640x384 1 frisbee, 101.0ms
Speed: 4.0ms preprocess, 101.0ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 frisbee, 41.8ms
Speed: 7.5ms preprocess, 41.8ms inference, 0.9ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 frisbee, 38.4ms
Speed: 3.7ms preprocess, 38.4ms inference, 0.9ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 40.4ms
Speed: 5.1ms preprocess, 40.4ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 37.5ms
Speed: 6.1ms preprocess, 37.5ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 39.7ms
Speed: 2.3ms preprocess, 39.7ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 36.6ms
Speed: 2.3ms preprocess, 36.6ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 28.7ms
Speed: 2.8ms preprocess, 28.7ms inference, 2.2ms 

In [15]:
test_img = "../../assets/demo.jpg"

# read the image and show in imshow
import cv2
image = cv2.imread(test_img)
cv2.imshow("Test Image", image)
cv2.waitKey(0)



error: OpenCV(4.10.0) D:\a\opencv-python\opencv-python\opencv\modules\highgui\src\window.cpp:1301: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'cvShowImage'


In [11]:
# predict on the image
model = YOLO("yolo26s_trained.pt")
results = model.predict(
    source="test_img_3.jpg",
    conf=0.6,
    save=True,
    save_txt=True,
    save_conf=True,
    device='0'
)

# show the image with bounding boxes
import matplotlib.pyplot as plt
import numpy as np
# Convert the first result to a numpy array
annotated_image = results[0].plot()
# Convert BGR to RGB for matplotlib
annotated_image_rgb = cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB)
# Display the image
plt.imshow(annotated_image_rgb)
plt.axis('off')  # Hide axes
plt.title("Predicted Image")
plt.show()

# save the annotated image
output_image_path = "../assets/annotated_image.jpg"
cv2.imwrite(output_image_path, annotated_image)



image 1/1 c:\Users\BurnD\Desktop\Flying_plate_tracker\yolo26_joel\training\test_img_3.jpg: 640x640 1 frisbee, 107.1ms
Speed: 8.1ms preprocess, 107.1ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 640)
Results saved to C:\Users\BurnD\Desktop\Flying_plate_tracker\yolo26_joel\training\runs\detect\predict3
1 label saved to C:\Users\BurnD\Desktop\Flying_plate_tracker\yolo26_joel\training\runs\detect\predict3\labels


<Figure size 640x480 with 1 Axes>

False

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Load the trained model
model = YOLO("yolo26s_trained.pt")

# Run validation to get metrics
metrics = model.val(
    data="../data/data.yaml",
    split="val",
    device="cuda"
)

print(f"mAP50:      {metrics.box.map50:.4f}")
print(f"mAP50-95:   {metrics.box.map:.4f}")
print(f"Precision:  {metrics.box.mp:.4f}")
print(f"Recall:     {metrics.box.mr:.4f}")

# Display the results curves
val_dir = metrics.save_dir
for img_name in ["results.png", "PR_curve.png", "F1_curve.png", "confusion_matrix.png"]:
    img_path = val_dir / img_name
    if img_path.exists():
        img = mpimg.imread(str(img_path))
        plt.figure(figsize=(12, 6))
        plt.imshow(img)
        plt.axis("off")
        plt.title(img_name)
        plt.show()
